# CIFAR-10 RGB vs Grayscale Analysis 
This notebook explores the performance difference of a CNN when trained on standard RGB CIFAR-10 images versus Grayscale images vs custom Greyscales. We use TensorBoard to monitor and compare the metrics.

In [1]:
import torch
from torchvision import transforms
import time
import os

from helper import *

torch.set_num_threads(8)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch version:', torch.__version__)
print('Device:', device)


PyTorch version: 2.12.0+cpu
Device: cpu


## 3. Data Preparation
CIFAR-10 is naturally RGB. We will tranform each We will take 4000 images from each 10 classes.

In [ ]:
IMAGE_SIZE = 32
BATCH_SIZE = 128
NUM_WORKERS = 2


transform_rgb = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_custom_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    LinearCombineChannel(0.36, 0.28, 0.36),
    transforms.Normalize((0.5,), (0.5,))
])

transform_custom_exclusive_gray = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    LinearCombineChannel(0.402, 0.255, 0.342),
    transforms.Normalize((0.5,), (0.5,))
])


train_loader_rgb, test_loader_rgb, class_names = create_dataloaders(transform_rgb, BATCH_SIZE, NUM_WORKERS, subset_per_class=4000)
train_loader_gray, test_loader_gray, _ = create_dataloaders(transform_gray, BATCH_SIZE, NUM_WORKERS, subset_per_class=4000)
train_loader_custom_gray, test_loader_custom_gray, _ = create_dataloaders(transform_custom_gray, BATCH_SIZE, NUM_WORKERS, subset_per_class=4000)
train_loader_custom_exclusive_gray, test_loader_custom_exclusive_gray, _ = create_dataloaders(transform_custom_exclusive_gray, BATCH_SIZE, NUM_WORKERS,subset_per_class=4000)

num_classes = len(class_names)
print('Classes:', class_names)


## 5. Experiment: RGB vs Grayscale

In [3]:
def run_exper(name, model, train_loader, test_loader, epochs=1):
    log_dir = os.path.join('runs', f'cifar10_{name}_{int(time.time())}')
    os.makedirs(log_dir, exist_ok=True)
    set_seed(42)  # re-set seed for deterministic execution within the thread
    _, acc = run_training(model, train_loader, test_loader, log_dir, device, IMAGE_SIZE, epochs)
    return name, acc


## 6. Experiment

In [ ]:
custom_experiments = [
    ("RGB", create_model_rgb(num_classes), train_loader_rgb, test_loader_rgb),
    ("Grey", create_model_single_channel(num_classes), train_loader_gray, test_loader_gray),
    ("CustomGray", create_model_single_channel(num_classes), train_loader_custom_gray, test_loader_custom_gray),
    ("CustomExclusiveGray", create_model_single_channel(num_classes), train_loader_custom_exclusive_gray, test_loader_custom_exclusive_gray)
]


for exp in custom_experiments:
    name, acc = run_exper(*exp,15)


In [ ]:
#  To display the tensorboard  , download the tensorboard library and run the code below

#tensorboard --logdir runs --bind_all 